In [19]:
import pathlib
home = pathlib.Path.home()
data_path = home / 'projects' / 'Markush' / 'data'
cpc_mcf_path = data_path / 'US_Grant_CPC_MCF_Text_2025-07-01.zip'

In [20]:
import zipfile
import io
from tqdm import tqdm

def extract_from_zip(zip_path: str, folder_name: str, output_path: str, cpc_subclasses: set):
    with zipfile.ZipFile(zip_path, 'r') as zf:
        prefix = folder_name.rstrip('/') + '/'
        # List all .txt files under that folder
        txt_files = [
            name for name in zf.namelist()
            if name.startswith(prefix) and name.lower().endswith('.txt')
        ]

        if not txt_files:
            print(f"Didn't find folder '{folder_name}' or no .txt files in it.")
            return

        unique_patent_num = set()

        # Iterate through each TXT file with progress feedback
        for txt_name in tqdm(txt_files,
                              desc='Processing files',
                              total=len(txt_files),
                              unit='file'):
            with zf.open(txt_name) as raw_f:
                # Wrap the raw binary stream to text for line-by-line reading
                f = io.TextIOWrapper(raw_f, encoding='utf-8', errors='ignore')
                for line in f:
                    # Remove trailing newline characters
                    line = line.rstrip('\n')
                    # Check if line is long enough and the 19-22th character is in
                    if len(line) >= 22 and line[18:22] in cpc_subclasses:
                        # Extract characters 11 through 18 (indices 10–17)
                        unique_patent_num.add(line[10:18])

        with open(output_path, 'w', encoding='utf-8') as out_f:
            for value in sorted(unique_patent_num):
                out_f.write(value + '\n')

    print(f"Finished. {len(unique_patent_num)} unique entries saved to '{output_path}'")

In [21]:
cpc_subclasses = (
    # 'C07C', 'C07D', 'C07F', 'C07G', 'C07H', 'C07K', 'C08G', 
    # 'A61K', 'A61P', 
    # 'A61K  31/00    ',
    # 'B01J'
    'A61K'
                  )

extract_from_zip(cpc_mcf_path, 'US_Grant_CPC_MCF_Text_2025-07-01', data_path / 'drug_patent_no.txt', cpc_subclasses)

Processing files: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 247/247 [00:20<00:00, 12.15file/s]


Finished. 321765 unique entries saved to 'chem_patent_num.txt'
